### 1. Yếu tố Tài chính
Ngân sách sản xuất (budget) và doanh thu phòng vé (revenue) có mối quan hệ như thế nào với rating trung bình của phim do người dùng MovieLens đánh giá?

#### a. Phân tích tương quan

In [ ]:
# --- BƯỚC TÍNH HỆ SỐ TƯƠNG QUAN (ĐÃ ĐIỀU CHỈNH) ---

# 1. Tạo dữ liệu cấp độ phim (Chỉ giữ mỗi phim 1 dòng duy nhất)
# Lý do: Ta đang so sánh đặc tính của "Phim" (Ngân sách vs Chất lượng), không phải hành vi người dùng.
df_movie_scope = full_data.drop_duplicates(subset=['MovieID'])

# 2. Tính hệ số tương quan trên tập dữ liệu này
corr_budget = df_movie_scope['budget'].corr(df_movie_scope['avg_rating'])
corr_revenue = df_movie_scope['revenue'].corr(df_movie_scope['avg_rating'])

# 3. In kết quả
print(f"Hệ số tương quan (Pearson) trên {df_movie_scope.shape[0]} bộ phim:")
print(f"- Budget  vs avg_rating: {corr_budget:.4f}")
print(f"- Revenue vs avg_rating: {corr_revenue:.4f}")

# Giải thích nhanh kết quả
if corr_budget < 0:
    print("-> Kết luận: Ngân sách càng cao, điểm đánh giá trung bình càng có xu hướng GIẢM.")
else:
    print("-> Kết luận: Ngân sách càng cao, điểm đánh giá trung bình càng TĂNG.")

Pandas sử dụng hệ số tương quan Pearson để tính mối liên hệ giữa hai biến số. Công thức tổng quát như sau:
$$
r_{XY} = 
\frac{\displaystyle \sum_{i=1}^{n} (X_i - \bar{X})(Y_i - \bar{Y})}
{\sqrt{\displaystyle \sum_{i=1}^{n} (X_i - \bar{X})^2 } \;
 \sqrt{\displaystyle \sum_{i=1}^{n} (Y_i - \bar{Y})^2 }}
$$

#### b. Trực quan hóa

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(16,6))

sns.regplot(x='budget', y='avg_rating', data=df_movie_scope, ax=axes[0], scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
axes[0].set_xscale('log')
axes[0].set_title('Ngân sách và Đánh giá')

sns.regplot(x='revenue', y='avg_rating', data=df_movie_scope, ax=axes[1], scatter_kws={'alpha':0.3}, line_kws={'color':'green'})
axes[1].set_xscale('log')
axes[1].set_title('Doanh thu và Đánh giá')

plt.tight_layout()
plt.savefig('images/correlation_analysis.png')
plt.show()


In [ ]:
# --- BƯỚC 1: PHÂN NHÓM NGÂN SÁCH (BUDGET BUCKETS) ---
# Chia ngân sách thành 4 nhóm chiến lược
bins = [0, 10000000, 50000000, 100000000, float('inf')]
labels = ['Micro/Indie\n(<$10M)', 'Mid-Budget\n($10M-$50M)', 'High-Budget\n($50M-$100M)', 'Blockbuster\n(>$100M)']

# Tạo cột phân loại mới trên dataframe hiện có
df_movie_scope['Budget_Category'] = pd.cut(df_movie_scope['budget'], bins=bins, labels=labels)

# Tính điểm trung bình cho từng nhóm để so sánh
group_stats = df_movie_scope.groupby('Budget_Category')['avg_rating'].agg(['mean', 'count']).sort_values('mean', ascending=False)

print("--- THỐNG KÊ ĐIỂM SỐ THEO NHÓM NGÂN SÁCH ---")
print(group_stats)

# --- BƯỚC 2: TRỰC QUAN HÓA PHI TUYẾN TÍNH ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Biểu đồ 1: Boxplot (So sánh phân phối điểm số)
sns.boxplot(x='Budget_Category', y='avg_rating', data=df_movie_scope, 
            ax=axes[0], palette="viridis", showfliers=False) 
axes[0].set_title('Phân phối Chất lượng phim theo quy mô Ngân sách', fontsize=14, fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Điểm trung bình (MovieLens)')
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# Biểu đồ 2: Scatter Plot với đường cong LOWESS (Non-linear trend)
sns.regplot(x='budget', y='avg_rating', data=df_movie_scope, 
            ax=axes[1], 
            scatter_kws={'alpha': 0.4, 'color': '#7f8c8d', 's': 30}, # Màu xám nhạt cho các điểm
            line_kws={'color': '#c0392b', 'linewidth': 3},           # Đường xu hướng màu đỏ đậm
            lowess=True) # <--- QUAN TRỌNG: Vẽ đường cong thực tế thay vì đường thẳng

axes[1].set_title('Xu hướng thực tế: Tiền càng nhiều, Điểm càng giảm?', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Ngân sách ($) - Thang Logarit')
axes[1].set_ylabel('Điểm trung bình (MovieLens)')
axes[1].set_xscale('log') # Dùng thang Log để giãn cách dữ liệu cho dễ nhìn
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('images/budget_vs_quality.png')
plt.show()

#### c. Phân cụm K-Means

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re


# 1. LOAD AND PREPARE DATA
# ------------------------
try:
    # Ratings
    ratings = pd.read_csv('datasets/movie_len_1M/ratings.dat', sep='::', names=['UserID', 'MovieID', 'Rating', 'Timestamp'], engine='python', encoding='latin-1')
    movie_targets = ratings.groupby('MovieID')['Rating'].mean().reset_index()
    movie_targets.columns = ['MovieID', 'avg_rating']

    # Movies (MovieLens)
    movies = pd.read_csv('datasets/movie_len_1M/movies.dat', sep='::', names=['MovieID', 'Title', 'Genres'], engine='python', encoding='latin-1')
    movies['clean_title'] = movies['Title'].apply(lambda x: re.sub(r'\s\(\d{4}\)', '', x).strip().lower())

    # TMDB Movies
    tmdb_movies = pd.read_csv('datasets/tmdb_5000/tmdb_5000_movies.csv')
    tmdb_movies['clean_title'] = tmdb_movies['title'].apply(lambda x: str(x).strip().lower())
    tmdb_movies['release_date'] = pd.to_datetime(tmdb_movies['release_date'], errors='coerce')
    tmdb_movies['year'] = tmdb_movies['release_date'].dt.year

    # Merge
    df_ml = pd.merge(movies, movie_targets, on='MovieID', how='inner')
    df_full = pd.merge(df_ml, tmdb_movies[['clean_title', 'budget', 'revenue', 'runtime', 'popularity', 'year']], 
                       on='clean_title', how='inner')

    # Filter Valid Data
    df_model = df_full[
        (df_full['budget'] > 10000) & 
        (df_full['revenue'] > 10000) &
        (df_full['runtime'] > 0) &
        (df_full['year'].notna())
    ].copy()

    # Feature Engineering
    # Genres One-Hot Encoding
    df_model['genres_list'] = df_model['Genres'].apply(lambda x: x.split('|'))
    mlb = MultiLabelBinarizer()
    genres_encoded = pd.DataFrame(mlb.fit_transform(df_model['genres_list']), columns=mlb.classes_, index=df_model.index)

    # Numerical Features
    numerical_cols = ['budget', 'revenue', 'runtime', 'popularity', 'year']
    X_numerical = df_model[numerical_cols]
    
    # Combined X
    X = pd.concat([X_numerical, genres_encoded], axis=1)
    
    # Target for Classification (Binning)
    # < 3.0: Low (0), 3.0 - 3.8: Medium (1), >= 3.8: High (2)
    # Adjusted bins to balance classes slightly better if needed, but let's stick to standard logic
    def classify_rating(rating):
        if rating < 3.0: return 0 # Low
        elif rating < 3.7: return 1 # Medium (MovieLens average is around 3.5)
        else: return 2 # High

    y_class = df_model['avg_rating'].apply(classify_rating)
    class_names = ['Low (<3.0)', 'Medium (3.0-3.7)', 'High (>=3.7)']

    print(f"Data ready. Shape: {X.shape}")
    print("Class distribution:\n", y_class.value_counts())

except Exception as e:
    print(f"Error preparing data: {e}")

# ------------------------------------------------------------
# 2. CLUSTERING (K-Means Rewritten and Improved)
# ------------------------------------------------------------
print("\n=== 2. Improved K-Means Clustering ===")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# 2.1 Scale toàn bộ feature
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2.2 Tìm số cụm tối ưu bằng Elbow + Silhouette
inertia_list = []
silhouette_list = []
K_range = range(2, 8)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    preds = km.fit_predict(X_scaled)
    inertia_list.append(km.inertia_)
    silhouette_list.append(silhouette_score(X_scaled, preds))

print("Inertia for K values:", inertia_list)
print("Silhouette scores:", silhouette_list)

# 2.3 Chọn K=3 (dựa vào domain knowledge: low–mid–high movie type)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)
df_model['cluster'] = clusters

# 2.4 Thống kê các cụm
cluster_desc = df_model.groupby('cluster')[['avg_rating', 'budget', 'revenue', 
                                            'popularity', 'runtime']].mean()

print("\nCluster Description (Means for each cluster):")
print(cluster_desc)

# 2.5 PCA Visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    x=X_pca[:, 0], y=X_pca[:, 1],
    hue=clusters, palette='viridis', s=40, alpha=0.7
)
plt.title("K-Means Clustering Projection (PCA)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.savefig("images/kmeans_pca.png")
plt.show()

#### d. Linear Regression theo từng cụm

In [ ]:
# Chạy cell này đầu tiên (nếu chưa import)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set_style("whitegrid")
%matplotlib inline

Chuẩn bị biến và lựa chọn biến độc lập

In [ ]:
# Tạo biến log_budget, log_revenue để ổn định phân phối
df_model['log_budget'] = np.log1p(df_model['budget'])
df_model['log_revenue'] = np.log1p(df_model['revenue'])

# Các biến độc lập cơ bản (bạn có thể thêm/dớt genres, runtime, popularity...)
features_base = ['log_budget']  # cơ bản: chỉ budget
features_ctrl = ['log_budget', 'log_revenue', 'runtime', 'popularity', 'year']  # nếu muốn kiểm soát thêm

# Target
target = 'avg_rating'

Hàm chạy OLS cho từng cụm và in summary

In [ ]:
def run_cluster_ols(df, cluster_col='cluster', features=None, target='avg_rating', add_constant=True, save_summary=False):
    """
    Chạy OLS cho từng cluster. Trả về dict chứa kết quả statsmodels.
    """
    results = {}
    clusters = sorted(df[cluster_col].unique())
    for c in clusters:
        sub = df[df[cluster_col] == c].dropna(subset=features + [target])
        if sub.shape[0] < 10:
            print(f"Cluster {c}: quá ít mẫu ({sub.shape[0]}) — bỏ qua.")
            continue
        
        X = sub[features].copy()
        if add_constant:
            X = sm.add_constant(X)
        y = sub[target]
        
        model = sm.OLS(y, X).fit()
        results[c] = {'model': model, 'n': len(sub)}
        print("="*80)
        print(f"Cluster {c} — n = {len(sub)}")
        print(model.summary())
        print("\n")
        if save_summary:
            with open(f"ols_cluster_{c}_summary.txt", "w", encoding="utf-8") as f:
                f.write(model.summary().as_text())
    return results

# Ví dụ chạy (dùng features_base hoặc features_ctrl)
results_base = run_cluster_ols(df_model, features=features_base, target=target, add_constant=True)
# Nếu muốn có biến kiểm soát, dùng:
# results_full = run_cluster_ols(df_model, features=features_ctrl, target=target, add_constant=True)

Vẽ scatter + đường hồi quy cho từng cụm (log_budget vs avg_rating)

In [ ]:
# Scatter + regression line (seaborn regplot) cho từng cluster
clusters = sorted(df_model['cluster'].unique())
n = len(clusters)
fig, axes = plt.subplots(1, n, figsize=(6*n, 5), sharey=True)

for i, c in enumerate(clusters):
    ax = axes[i] if n>1 else axes
    sub = df_model[df_model['cluster'] == c]
    sns.regplot(x='log_budget', y='avg_rating', data=sub, scatter_kws={'s':20, 'alpha':0.4}, line_kws={'color':'red'}, ax=ax)
    ax.set_title(f'Cluster {c} (n={len(sub)})')
    ax.set_xlabel('log_budget')
    if i==0:
        ax.set_ylabel('avg_rating')

plt.suptitle('Budget (log) vs Rating theo từng cụm')
plt.tight_layout(rect=[0,0,1,0.95])
plt.savefig('images/clusterwise_budget_vs_rating.png', dpi=150)
plt.show()

Residual plots và kiểm tra cơ bản cho mỗi model

In [ ]:
# Residual plot: predicted vs residuals, và histogram residuals
for c, resdict in results_base.items():
    model = resdict['model']
    sub = df_model[df_model['cluster'] == c].dropna(subset=features_base + [target])
    X = sm.add_constant(sub[features_base])
    y = sub[target]
    preds = model.predict(X)
    residuals = y - preds
    
    fig, axes = plt.subplots(1, 2, figsize=(12,4))
    sns.scatterplot(x=preds, y=residuals, ax=axes[0], s=30, alpha=0.5)
    axes[0].axhline(0, color='red', linestyle='--')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Residuals')
    axes[0].set_title(f'Cluster {c} Residuals vs Predicted')
    
    sns.histplot(residuals, bins=25, kde=True, ax=axes[1])
    axes[1].set_title(f'Cluster {c} Residuals Distribution')
    plt.tight_layout()
    plt.savefig(f'images/cluster_{c}_residuals.png', dpi=150)
    plt.show()


So sánh hệ số (coefficients) giữa các cụm

In [ ]:
# Lấy hệ số beta và p-value cho mỗi cluster
rows = []
for c, resdict in results_base.items():
    model = resdict['model']
    params = model.params
    pvalues = model.pvalues
    rows.append({
        'cluster': c,
        'n': resdict['n'],
        'coef_log_budget': params.get('log_budget', np.nan),
        'pvalue_log_budget': pvalues.get('log_budget', np.nan),
        'r2': model.rsquared
    })

coef_df = pd.DataFrame(rows).sort_values('cluster')
print(coef_df)

# Vẽ bar plot hệ số
plt.figure(figsize=(8,4))
sns.barplot(x='cluster', y='coef_log_budget', data=coef_df)
plt.title('Coefficient of log_budget theo cluster')
plt.xlabel('Cluster')
plt.ylabel('Beta (log_budget)')
plt.axhline(0, color='black', linewidth=0.8)
plt.savefig('images/coef_log_budget_by_cluster.png', dpi=150)
plt.show()

In [ ]:
coef_df.to_csv('result/clusterwise_regression_results.csv', index=False)